<a href="https://colab.research.google.com/github/nakamura196/01_notebooks/blob/main/lora_ndc_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoRAで「本のタイトルからジャンルを当てるAI」を作る

図書館では、本に **NDC（日本十進分類法）** というジャンル番号を付けています。

| NDC | ジャンル |
|-----|----------|
| 0 | 総記（百科事典・情報学など） |
| 1 | 哲学・宗教 |
| 2 | 歴史・地理 |
| 3 | 社会科学（法律・経済・教育） |
| 4 | 自然科学（数学・物理・医学） |
| 5 | 技術・工学 |
| 6 | 産業（農業・商業・運輸） |
| 7 | 芸術・スポーツ |
| 8 | 言語 |
| 9 | 文学 |

**「本のタイトルだけを見てNDCを当てる」** ことを、LoRAでLLMに教えてみましょう。

データは **国立国会図書館サーチAPI** からリアルタイムで取得します。

## Step 0. セットアップ

In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes

In [ ]:
import html
import json
import random
import re
import time
import urllib.request
import xml.etree.ElementTree as ET
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"デバイス: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1. NDLサーチAPIから書誌データを取得

国立国会図書館サーチの **SRU API**（誰でも利用可能）を使って、
各ジャンルの本のタイトルとNDCコードを取得します。

APIの詳細: https://ndlsearch.ndl.go.jp/file/help/api/specifications/ndlsearch_api_sru.pdf

In [ ]:
# NDC名称
NDC_NAMES = {
    "0": "総記",
    "1": "哲学",
    "2": "歴史",
    "3": "社会科学",
    "4": "自然科学",
    "5": "技術・工学",
    "6": "産業",
    "7": "芸術・スポーツ",
    "8": "言語",
    "9": "文学",
}

NS_SRW = "http://www.loc.gov/zing/srw/"
NS_INNER = {
    "dc": "http://purl.org/dc/elements/1.1/",
    "dcndl": "http://ndl.go.jp/dcndl/terms/",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "dcterms": "http://purl.org/dc/terms/",
}


def fetch_ndl_books(ndc_digit, count=80, start=1):
    """
    NDLサーチSRU APIから指定NDCの書誌を取得する。
    ndc_digit: NDC1桁目 ("0"〜"9")
    count: 取得件数 (最大100)
    """
    base_url = "https://ndlsearch.ndl.go.jp/api/sru"
    query = f'ndc="{ndc_digit}*"'
    params = (
        f"?operation=searchRetrieve"
        f"&query={urllib.parse.quote(query)}"
        f"&maximumRecords={count}"
        f"&startRecord={start}"
        f"&recordSchema=dcndl"
    )
    url = base_url + params

    req = urllib.request.Request(url)
    with urllib.request.urlopen(req, timeout=30) as resp:
        xml_text = resp.read().decode("utf-8")

    root = ET.fromstring(xml_text)

    books = []
    for record in root.findall(f".//{{{NS_SRW}}}record"):
        # recordData の中身はHTMLエスケープされたXML
        rd = record.find(f"{{{NS_SRW}}}recordData")
        if rd is None or rd.text is None:
            continue
        inner_xml = html.unescape(rd.text)

        try:
            inner_root = ET.fromstring(inner_xml)
        except ET.ParseError:
            continue

        # タイトル取得
        title_el = inner_root.find(".//dcterms:title", NS_INNER)
        if title_el is None or not title_el.text:
            continue
        title = title_el.text.strip()

        # タイトルが短すぎる/長すぎるものは除外
        if len(title) < 3 or len(title) > 80:
            continue

        # NDC取得（dc:subject で datatype に "NDC" を含むもの）
        ndc_code = ""
        for subj in inner_root.findall(".//dc:subject", NS_INNER):
            dt = subj.get(f"{{{NS_INNER['rdf']}}}datatype", "")
            if "NDC" in dt and subj.text:
                ndc_code = subj.text.strip()
                break

        books.append({
            "title": title,
            "ndc": ndc_digit,
            "ndc_full": ndc_code,
            "ndc_name": NDC_NAMES[ndc_digit],
        })

    return books


# 全NDCカテゴリから書誌を取得
print("📚 NDLサーチAPIからデータ取得中...")
all_books = []
for digit in "0123456789":
    books = fetch_ndl_books(digit, count=80)
    all_books.extend(books)
    print(f"  NDC {digit} ({NDC_NAMES[digit]}): {len(books)}件")
    time.sleep(1)  # API負荷軽減

print(f"\n合計: {len(all_books)}件")

### 取得したデータの例を見てみましょう

In [ ]:
# ジャンルごとにタイトルの例を表示
print("📖 各ジャンルのタイトル例:\n")
for digit in "0123456789":
    examples = [b["title"] for b in all_books if b["ndc"] == digit][:3]
    print(f"  NDC {digit} ({NDC_NAMES[digit]}):")
    for t in examples:
        print(f"    ・{t}")
    print()

### 学習用・テスト用に分割

In [ ]:
random.shuffle(all_books)

# 各カテゴリからテスト用に5件ずつ確保
test_books = []
train_books = []
test_count_per_class = 5
test_counter = Counter()

for book in all_books:
    if test_counter[book["ndc"]] < test_count_per_class:
        test_books.append(book)
        test_counter[book["ndc"]] += 1
    else:
        train_books.append(book)

print(f"学習用: {len(train_books)}件")
print(f"テスト用: {len(test_books)}件（各ジャンル{test_count_per_class}件 × 10ジャンル）")

## Step 2. プロンプト設計

モデルに「本のタイトルを見てNDCの1桁目を答える」というタスクを与えます。

In [ ]:
NDC_LIST = "\n".join([f"{k}: {v}" for k, v in NDC_NAMES.items()])


def make_prompt(book, include_answer=False):
    """NDC分類のプロンプトを生成する"""
    text = (
        f"以下の本のタイトルから、NDC（日本十進分類法）の1桁目を答えてください。\n"
        f"\n"
        f"【NDC一覧】\n"
        f"{NDC_LIST}\n"
        f"\n"
        f"【タイトル】{book['title']}\n"
        f"【NDC】"
    )
    if include_answer:
        text += f" {book['ndc']}"
    return text


# プロンプトの例を表示
print("--- プロンプト例 ---")
print(make_prompt(test_books[0]))
print(f"(正解: {test_books[0]['ndc']} = {test_books[0]['ndc_name']})")
print("-------------------")

## Step 3. モデル読み込み

In [ ]:
MODEL_NAME = "llm-jp/llm-jp-3-1.8b"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"モデル: {MODEL_NAME}")
print(f"パラメータ数: {model.num_parameters():,}")

## Step 4. 評価関数

In [ ]:
def evaluate(model, books, label=""):
    """モデルにNDC分類させて正答率を計算する"""
    correct = 0
    results = []
    model.eval()

    with torch.no_grad():
        for i, book in enumerate(books):
            prompt = make_prompt(book, include_answer=False)
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            generated = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            ).strip()

            # 最初に出現する数字(0-9)を予測値とする
            predicted = "?"
            for ch in generated:
                if ch in "0123456789":
                    predicted = ch
                    break

            is_correct = predicted == book["ndc"]
            if is_correct:
                correct += 1

            results.append({
                "title": book["title"],
                "predicted": predicted,
                "predicted_name": NDC_NAMES.get(predicted, "?"),
                "answer": book["ndc"],
                "answer_name": book["ndc_name"],
                "correct": is_correct,
                "raw_output": generated[:40],
            })

    accuracy = correct / len(books) * 100
    print(f"📊 {label} 正答率: {correct}/{len(books)} = {accuracy:.1f}%")
    return accuracy, results

## Step 5. 【学習前】テスト

素のモデルは、本のタイトルからジャンルを当てられるでしょうか？

In [ ]:
acc_before, results_before = evaluate(model, test_books, label="学習前")

print(f"\n(参考: 10択ランダムの期待値は10.0%)\n")
print("--- 回答例 ---")
for r in results_before[:15]:
    mark = "✅" if r["correct"] else "❌"
    print(f"  {mark} 『{r['title'][:25]}』")
    print(f"     予測: {r['predicted']}({r['predicted_name']})  正解: {r['answer']}({r['answer_name']})  生成: {r['raw_output'][:30]}")

## Step 6. LoRA設定と適用

モデル全体は凍結し、Attention層に小さな「アダプター」を挿入します。

```
モデル本体 (18億パラメータ)  →  凍結（変更しない）
      ↓
LoRAアダプター (数百万パラメータ)  →  ここだけ学習！
```

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"全パラメータ:  {total:>14,}")
print(f"学習対象:      {trainable:>14,} ({trainable/total*100:.2f}%)")
print(f"\n→ 全体の約 {total//trainable} 分の1 だけ学習します")

## Step 7. 学習データ準備

In [ ]:
train_texts = [make_prompt(b, include_answer=True) for b in train_books]
train_dataset = Dataset.from_dict({"text": train_texts})

print(f"学習データ: {len(train_dataset)}件")
print(f"\n--- 学習データの例 ---")
print(train_texts[0])
print("----------------------")

## Step 8. LoRA学習の実行

In [ ]:
training_args = SFTConfig(
      output_dir="./lora_ndc_output",
      per_device_train_batch_size=4,
      gradient_accumulation_steps=4,
      learning_rate=5e-4,
      num_train_epochs=5,
      logging_steps=10,
      save_strategy="epoch",
      bf16=True,
      dataset_text_field="text",
      report_to="none",
  )

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args
)

print("🎓 学習開始...\n")
trainer.train()
print("\n学習完了！")

## Step 9. 【学習後】テスト

In [ ]:
acc_after, results_after = evaluate(model, test_books, label="学習後")

print(f"\n--- 回答例 ---")
for r in results_after[:15]:
    mark = "✅" if r["correct"] else "❌"
    print(f"  {mark} 『{r['title'][:25]}』")
    print(f"     予測: {r['predicted']}({r['predicted_name']})  正解: {r['answer']}({r['answer_name']})  生成: {r['raw_output'][:30]}")

## Step 10. 結果の比較

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

print("=" * 50)
print("📋 結果まとめ")
print("=" * 50)
print(f"  学習前:   {acc_before:5.1f}%")
print(f"  学習後:   {acc_after:5.1f}%")
print(f"  改善幅:  +{acc_after - acc_before:.1f}ポイント")
print(f"  ランダム:  10.0%（10クラス）")
print(f"  学習量:    全パラメータの{trainable/total*100:.2f}%のみ")
print("=" * 50)

# --- グラフ ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左: 正答率
bars = axes[0].bar(
    ["Before LoRA", "After LoRA", "Random"],
    [acc_before, acc_after, 10.0],
    color=["#4a90d9", "#e8513d", "#cccccc"],
    edgecolor="black", linewidth=0.5,
)
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("NDC Classification Accuracy")
axes[0].set_ylim(0, 100)
for bar, val in zip(bars, [acc_before, acc_after, 10.0]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{val:.1f}%", ha="center", fontsize=14, fontweight="bold")

# 右: ジャンル別正答率 (学習後)
ndc_accs = {}
for d in "0123456789":
    cat_results = [r for r in results_after if r["answer"] == d]
    if cat_results:
        ndc_accs[d] = sum(r["correct"] for r in cat_results) / len(cat_results) * 100

labels = [f"{d}:{NDC_NAMES[d][:4]}" for d in ndc_accs]
values = list(ndc_accs.values())
colors = plt.cm.tab10(range(len(labels)))
axes[1].barh(labels, values, color=colors, edgecolor="black", linewidth=0.5)
axes[1].set_xlabel("Accuracy (%)")
axes[1].set_title("Accuracy by NDC Category (After LoRA)")
axes[1].set_xlim(0, 105)
for i, v in enumerate(values):
    axes[1].text(v + 1, i, f"{v:.0f}%", va="center", fontsize=10)

plt.suptitle("LoRA Fine-Tuning: NDC Classification from Book Titles", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("lora_ndc_result.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 11. 学習前後の比較テーブル

In [ ]:
print("学習前後の予測比較")
print("=" * 80)
print(f"{'タイトル':<30} {'正解':>6} {'学習前':>8} {'学習後':>8}")
print("-" * 80)
for rb, ra in zip(results_before, results_after):
    title = rb["title"][:28]
    ans = f"{rb['answer']}({rb['answer_name'][:4]})"
    bef = f"{rb['predicted']}({'✅' if rb['correct'] else '❌'})"
    aft = f"{ra['predicted']}({'✅' if ra['correct'] else '❌'})"
    print(f"  {title:<28} {ans:>8} {bef:>8} {aft:>8}")

## Step 12. 自分の好きな本で試してみよう！

学習済みモデルに、好きな本のタイトルを入れて分類させてみましょう。

In [ ]:
def predict_ndc(model, title):
    """タイトルからNDCを予測する"""
    book = {"title": title, "ndc": "?", "ndc_name": "?"}
    prompt = make_prompt(book, include_answer=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicted = "?"
    for ch in generated:
        if ch in "0123456789":
            predicted = ch
            break

    name = NDC_NAMES.get(predicted, "不明")
    return predicted, name, generated[:40]


# --- 試してみよう！ ---
test_titles = [
    "吾輩は猫である",
    "相対性理論入門",
    "日本経済の構造改革",
    "フランス料理の基本技法",
    "はじめてのPython入門",
    "万葉集を読む",
    "西洋美術史",
    "憲法判例百選",
    "英語の語源辞典",
    "鉄道の歴史と未来",
]

print("🔮 好きな本のタイトルでNDC予測！\n")
for title in test_titles:
    ndc, name, raw = predict_ndc(model, title)
    print(f"  『{title}』→ NDC {ndc} ({name})")

In [ ]:
# 自分でタイトルを入力して試す場合はここを編集してください
my_title = "ここに本のタイトルを入れてください"
ndc, name, raw = predict_ndc(model, my_title)
print(f"『{my_title}』→ NDC {ndc} ({name})")
print(f"(モデルの生成: {raw})")

## まとめ

### やったこと

1. **国立国会図書館サーチAPI** からリアルタイムで書誌データを取得
2. 小型日本語LLM（llm-jp-3-1.8b）に **LoRA** で「タイトル→NDC分類」を学習
3. 学習前後の正答率を比較し、**ジャンル別の得意・不得意** も分析
4. 好きな本のタイトルで実際に予測を体験

### わかったこと

- 素のモデルはタイトルからジャンルを当てるのが苦手（ランダムに近い）
- LoRAで **全パラメータの0.5%だけ** を学習させるだけで大幅に改善
- LoRAは「知識」ではなく **「タスクの解き方」** を教える技術
- 実際の図書館業務でも、分類作業の支援に応用できる可能性がある

### 発展

- NDC3桁分類（913: 日本の小説、490: 医学 など）に挑戦するとより実用的
- タイトルだけでなく著者・出版社の情報も加えれば精度はさらに上がる
- RAG（検索拡張生成）と組み合わせて、類似書の分類を参照させる方法もある